<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 06 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Joining Data in Apache Doris</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:920px;margin:0">Choose a Join from the rows the result must retain, protect the intended result grain, and connect logical Join semantics to Doris physical execution.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">INNER · OUTER · SEMI · ANTI · Hash · Nested Loop · Broadcast · Bucket Shuffle · Runtime Filter</span>
</div>

This lab continues with `events_modelled`, whose grain is one row per original event. It creates its own deterministic dimension and small Join cases, so no additional public dataset, S3 object, or credential is required.


### Initialize the Lab

Run the next cell before Section 1. It loads the shared course helper and creates the `lab` object used by every later cell. Run it again after restarting the Jupyter kernel. It does not start Docker or change Doris data.


In [1]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


## 1. Build a controlled product relationship from the baseline

A deterministic `dim_products` table is generated locally from the baseline product identifiers, with a small number of controlled unmatched test rows. No second public dataset or external download is required.

The table keeps one row per product by using a Unique Key model. Category and brand are deterministic teaching attributes calculated from `product_id`. It contains 204,231 product rows; the following three identifiers are selected later only as controlled examples, not as the complete dimension:

| `product_id` | In `events_modelled` | In `dim_products` | Relationship |
|---:|---|---|---|
| `1004767` | Yes | Yes | Matched |
| `1005115` | Yes | No | Event only |
| `999999999` | No | Yes | Dimension only |

Product `1005115` is deliberately omitted while product `999999999` is deliberately added. These controlled exceptions make both directions of an unmatched relationship observable.


In [ ]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.execute("DROP TABLE IF EXISTS dim_products")
lab.execute("""
CREATE TABLE dim_products (
    product_id BIGINT NOT NULL,
    category VARCHAR(32) NOT NULL,
    brand VARCHAR(32) NOT NULL
)
UNIQUE KEY(product_id)
DISTRIBUTED BY HASH(product_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")

lab.insert("""
INSERT INTO dim_products (product_id, category, brand)
SELECT DISTINCT
    product_id,
    CONCAT('category_', MOD(product_id, 8) + 1),
    CONCAT('brand_', MOD(product_id, 20) + 1)
FROM events_modelled
WHERE product_id <> 1005115
UNION ALL
SELECT 999999999, 'category_unused', 'brand_unused'
""", title="Build the deterministic product dimension")



**Expected result:** the INSERT reports 204,231 affected rows. Each `product_id` appears once in `dim_products`, so one matched event row can receive at most one category and brand. Re-running the cell drops and rebuilds the table, so it does not duplicate rows. The controlled relationships shown above are queried in the next section.


## 2. Choose the Join type from the rows the result must retain

The following queries deliberately limit the event input to two product identifiers so the retained rows are easy to see. This `WHERE` clause does not limit the contents of `dim_products`: it selects matched product `1004767` and event-only product `1005115` as a controlled comparison.

`INNER JOIN` retains only matching pairs. `LEFT OUTER JOIN` also retains event rows without a product definition and fills the right-side attributes with `NULL`.


In [ ]:
lab.sql("""
SELECT e.product_id, d.category, COUNT(*) AS retained_event_rows
FROM events_modelled e
INNER JOIN dim_products d ON e.product_id = d.product_id
WHERE e.product_id IN (1004767, 1005115)
GROUP BY e.product_id, d.category
ORDER BY e.product_id
""", title="Rows retained by INNER JOIN")

lab.sql("""
SELECT e.product_id, d.category, COUNT(*) AS retained_event_rows
FROM events_modelled e
LEFT OUTER JOIN dim_products d ON e.product_id = d.product_id
WHERE e.product_id IN (1004767, 1005115)
GROUP BY e.product_id, d.category
ORDER BY e.product_id
""", title="Rows retained by LEFT OUTER JOIN");


`LEFT SEMI JOIN` and `LEFT ANTI JOIN` return only columns from the retained left side. The queries therefore display the retained `product_id` and event count rather than right-side attributes. Semi keeps the matched product; Anti keeps the event-only product.


In [ ]:
lab.sql("""
SELECT e.product_id, COUNT(*) AS retained_event_rows
FROM events_modelled e
LEFT SEMI JOIN dim_products d ON e.product_id = d.product_id
WHERE e.product_id IN (1004767, 1005115)
GROUP BY e.product_id
""", title="Rows retained by LEFT SEMI JOIN")

lab.sql("""
SELECT e.product_id, COUNT(*) AS retained_event_rows
FROM events_modelled e
LEFT ANTI JOIN dim_products d ON e.product_id = d.product_id
WHERE e.product_id IN (1004767, 1005115)
GROUP BY e.product_id
""", title="Rows retained by LEFT ANTI JOIN");


**Expected result:** each short query makes its retention rule visible:

- `INNER JOIN` returns only `1004767`, with category `category_8` and 105,046 retained event rows. `1005115` does not appear because it has no matching dimension row.
- `LEFT OUTER JOIN` returns both identifiers. `1005115` retains 89,522 event rows, while its category is `NULL`.
- `LEFT SEMI JOIN` returns only matched identifier `1004767` and its 105,046 event rows.
- `LEFT ANTI JOIN` returns only event-only identifier `1005115` and its 89,522 event rows.

Semi and Anti are complementary for this controlled input: one keeps the matched subset and the other keeps the unmatched subset. RIGHT and FULL OUTER JOIN apply corresponding retention rules from the other side. CROSS JOIN has no Join condition and produces a Cartesian product, so it is not run on the baseline.


## 3. Produce a fact-dimension analytical result

The main analytical query now enriches purchase events with the product category and then aggregates at category-and-region grain. Because `dim_products` has one row per `product_id`, the Join does not multiply matched event rows. `INNER JOIN` intentionally excludes events whose product has no dimension row.


In [ ]:
lab.sql("""
SELECT
    d.category,
    e.region,
    COUNT(*) AS purchase_events,
    SUM(e.revenue) AS purchase_revenue
FROM events_modelled e
INNER JOIN dim_products d
    ON e.product_id = d.product_id
WHERE e.event_time >= '2020-03-03 00:00:00'
  AND e.event_time <  '2020-03-04 00:00:00'
  AND e.event_type = 'purchase'
GROUP BY d.category, e.region
ORDER BY purchase_revenue DESC, d.category, e.region
LIMIT 12
""", title="Purchase revenue by product category and region");


**Expected result:** twelve category-and-region groups are displayed in descending revenue order. No category is `NULL` because the INNER JOIN retains only events with a matching dimension row. The result grain is now one row per `category + region` combination rather than one row per original event.


## 4. Predict how duplicate Join keys change the result grain

The controlled event table contains four source rows:

| `event_id` | `product_id` | Relationship prepared by the row |
|---:|---:|---|
| `1` | `10` | One matching product row |
| `2` | `20` | Two matching product-version rows |
| `3` | `30` | No matching product row |
| `4` | `NULL` | Missing Join key |

The controlled product table contains five rows:

| `product_id` | `version_label` | Relationship prepared by the row |
|---:|---|---|
| `10` | `current` | Matches event `1` once |
| `20` | `old` | First match for event `2` |
| `20` | `current` | Second match for event `2` |
| `40` | `unused` | Has no event row |
| `NULL` | `unknown` | Used in the next section to examine NULL matching |

A Join returns every matching pair and does not require the right-side Join key to be unique. Because `product_id = 20` appears once with `version_label = 'old'` and once with `version_label = 'current'`, event `2` matches both product rows and produces two result rows.

This row multiplication is valid Join behavior. Whether it is desirable depends on the expected result grain: it is correct when the result should contain every event–product-version relationship, but it must be resolved first when the result should contain exactly one enriched row per event.


In [18]:
lab.execute("DROP TABLE IF EXISTS join_event_cases")
lab.execute("DROP TABLE IF EXISTS join_product_cases")
lab.execute("""
CREATE TABLE join_event_cases (event_id BIGINT, product_id BIGINT NULL)
DUPLICATE KEY(event_id)
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.execute("""
CREATE TABLE join_product_cases (product_id BIGINT NULL, version_label VARCHAR(16))
DUPLICATE KEY(product_id)
DISTRIBUTED BY HASH(product_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("INSERT INTO join_event_cases VALUES (1, 10), (2, 20), (3, 30), (4, NULL)", title="Write controlled event rows")
lab.insert("INSERT INTO join_product_cases VALUES (10, 'current'), (20, 'old'), (20, 'current'), (40, 'unused'), (NULL, 'unknown')", title="Write controlled product rows")

lab.sql("""
SELECT e.event_id, e.product_id, d.version_label
FROM join_event_cases e
LEFT OUTER JOIN join_product_cases d
    ON e.product_id = d.product_id
ORDER BY e.event_id, d.version_label
""", title="One-to-many Join result");


event_id,product_id,version_label
1,10.0,current
2,20.0,current
2,20.0,old
3,30.0,NULL
4,nan,NULL


**Expected result:** four source event rows produce five matching-result rows. Event `2` appears twice because its `product_id = 20` matches both `old` and `current`. Events `3` and `4` remain because this is a LEFT OUTER JOIN, but their `version_label` is `NULL`. If the required result grain is one row per event, the product input must first be reduced to one row per `product_id`; if every product version is required, the five-row result is already correct.


## 5. Choose whether NULL Join keys can match

Ordinary equality returns unknown when either operand is `NULL`, so `NULL = NULL` does not form a matching pair. Doris also provides the NULL-safe equality operator `<=>`; it returns true when both operands are `NULL`. Use it only when the business relationship defines two missing keys as the same value.


In [20]:
lab.sql("""
SELECT join_condition, matching_pairs
FROM (
    SELECT 1 AS sequence, '=' AS join_condition, COUNT(*) AS matching_pairs
    FROM join_event_cases e INNER JOIN join_product_cases d
        ON e.product_id = d.product_id
    WHERE e.event_id = 4
    UNION ALL
    SELECT 2, '<=>', COUNT(*)
    FROM join_event_cases e INNER JOIN join_product_cases d
        ON e.product_id <=> d.product_id
    WHERE e.event_id = 4
) comparison
ORDER BY sequence
""", title="NULL Join-key comparison");


join_condition,matching_pairs
=,0
<=>,1


**Expected result:** `=` produces zero matching pairs and `<=>` produces one. NULL-aware Anti Join addresses the related `NOT IN`/Anti-Join case where `NULL` must be handled explicitly; it is introduced in the course without adding another large-table query here.


## 6. Connect the Join condition to the physical implementation

For an equi-condition, Doris can build a hash table from the right side and probe it with left-side rows. A pure non-equi condition cannot be used as a hash key, so Doris can choose a Nested Loop Join. Both examples use only the tiny controlled tables; the complete plans remain available under the disclosure below the summary.


In [ ]:
lab.compare_join_plans([
    ("Equi-condition", """
     EXPLAIN SHAPE PLAN
     SELECT COUNT(*)
     FROM join_event_cases e
     INNER JOIN join_product_cases d ON e.product_id = d.product_id
     """),
    ("Non-equi condition", """
     EXPLAIN SHAPE PLAN
     SELECT COUNT(*)
     FROM join_event_cases e
     INNER JOIN join_product_cases d ON e.product_id > d.product_id
     """),
], title="Join condition and physical implementation");


**Expected result:** the equi-condition uses a Hash Join; the non-equi condition uses a Nested Loop Join. In the Hash Join, the right side is the build side and the left side is the probe side. Nested Loop Join is more general but can compare many more row pairs, which is why the lab keeps its input deliberately small.


## 7. Read data movement from the plan and filtering from the Profile

Broadcast and Shuffle are most important in a multi-BE distributed cluster, where they determine how Join data moves between BE nodes. Broadcast copies the smaller input to the BE nodes executing the Join, while Shuffle redistributes rows according to the Join key.

This Lab uses a single-BE sandbox, so it does not perform a real cross-BE Shuffle. It demonstrates the distribution strategy and Exchange structure produced by the optimizer. The first plan lets the optimizer select Broadcast for the relatively small `dim_products` relation. The second plan deliberately uses the `[shuffle]` hint as a comparison plan; it is not a default recommendation. Because `dim_products` is already Hash bucketed by the Join key, the sandbox can retain that bucket layout and report Bucket Shuffle while redistributing the other input. Compare the plan evidence rather than elapsed time or network performance.

For a Hash Join, Doris can generate a Runtime Filter from build-side product identifiers and apply it to the probe-side Scan. This can reject event rows that cannot match before they reach the Join operator.


In [ ]:
lab.explain_plan("""
EXPLAIN SHAPE PLAN
SELECT COUNT(*)
FROM events_modelled e
INNER JOIN dim_products d ON e.product_id = d.product_id
WHERE d.category = 'category_1'
  AND e.event_time >= '2020-03-01 00:00:00'
  AND e.event_time <  '2020-03-02 00:00:00'
""", title="Default Join plan")

lab.explain_plan("""
EXPLAIN SHAPE PLAN
SELECT COUNT(*)
FROM events_modelled e
INNER JOIN [shuffle] dim_products d ON e.product_id = d.product_id
WHERE d.category = 'category_1'
  AND e.event_time >= '2020-03-01 00:00:00'
  AND e.event_time <  '2020-03-02 00:00:00'
""", title="Join plan with the [shuffle] hint");


**Expected result:** read the two raw plan trees from the Join operator toward their Scan operators:

1. In **Default Join plan**, find `hashJoin[INNER_JOIN broadcast]`. This identifies an Inner Hash Join whose smaller input is Broadcast.
2. On the same Hash Join line, find `build RFs:RF... product_id->[product_id]`. Doris builds one or more Runtime Filters from the build-side `product_id` values. Runtime Filter identifiers such as `RF0` may vary.
3. Under the `events_modelled` branch, find `PhysicalOlapScan[events_modelled] apply RFs: RF...`. The probe-side Scan receives those Runtime Filters, so rows that cannot match can be rejected before reaching the Join operator.
4. In **Join plan with the [shuffle] hint**, find `hashJoin[INNER_JOIN shuffleBucket]`. In the tested Doris 4.1.3 sandbox, the non-Broadcast plan reuses an existing Hash bucket layout on `product_id` and reports Bucket Shuffle.
5. At the end of the hinted plan, `Hint log` should list `[shuffle]` under `Used`, confirming that Doris accepted the hint.

A general Partition Shuffle redistributes both inputs by the Join key, while Bucket Shuffle retains one compatible Bucket layout and redistributes the other input. Colocate can avoid Join shuffle when both tables satisfy the same colocation contract. No elapsed time appears here because `EXPLAIN SHAPE PLAN` creates the distributed query plan without executing the query. The next cell executes the default Join and reads its Runtime Profile to see whether event rows were filtered before reaching the Join. Elapsed time from this single-node sandbox should not be used to rank multi-node Join strategies.


### Execute the Join and inspect probe-side filtering

`EXPLAIN SHAPE PLAN` describes the intended Runtime Filters but does not execute the query. Run the same default Join and inspect the actual Query Profile. The output below is a cropped **raw MergedProfile excerpt**, not a reformatted metrics table: it keeps the Join's `ProbeRows` and the `events_modelled` Scan's `RF... InputRows`, `RF... FilterRows`, `RowsProduced`, and `ScanRows` counters. Doris may create more than one Runtime Filter, so inspect each filter's own counters.


In [2]:
lab.show_join_runtime_filter_profile("""
SELECT COUNT(*)
FROM events_modelled e
INNER JOIN dim_products d ON e.product_id = d.product_id
WHERE d.category = 'category_1'
  AND e.event_time >= '2020-03-01 00:00:00'
  AND e.event_time <  '2020-03-02 00:00:00'
""", probe_table="events_modelled", title="Probe-side Join Profile excerpt")


Query result: COUNT(*)=13325


'HASH_JOIN_OPERATOR(nereids_id=371)(id=3):\n                - ProbeRows: sum 13.325K (13325), avg 3.331K (3331), max 4.051K (4051), min 2.501K (2501)\n           OLAP_SCAN_OPERATOR(nereids_id=346. table_name=events_modelled(events_modelled))(id=2):\n                - RowsProduced: sum 13.325K (13325), avg 3.331K (3331), max 4.051K (4051), min 2.501K (2501)\n                  - RF0 FilterRows: sum 0, avg 0, max 0, min 0\n                  - RF0 InputRows: sum 7.438K (7438), avg 1.859K (1859), max 2.629K (2629), min 171\n                  - RF1 FilterRows: sum 61.934K (61934), avg 15.483K (15483), max 18.743K (18743), min 12.176K (12176)\n                  - RF1 InputRows: sum 75.259K (75259), avg 18.814K (18814), max 22.794K (22794), min 14.952K (14952)\n                - ScanRows: sum 75.259K (75259), avg 18.814K (18814), max 22.794K (22794), min 14.952K (14952)'

**Expected result:** the query returns the count of matching events, followed by a short text excerpt with `HASH_JOIN_OPERATOR` and `OLAP_SCAN_OPERATOR(...table_name=events_modelled...)` headings.

Read the **`sum`** values across execution instances in this order:

1. Under the `events_modelled` Scan, `RF... InputRows` counts rows presented to that particular Runtime Filter, and `RF... FilterRows` counts rows it rejected. Match the two counters by RF identifier; do not add input counts from different filters because filters can run in sequence.
2. A positive `RF... FilterRows` on this probe-side Scan is execution evidence that some event rows were rejected before the Join. In the tested Doris 4.1.3 sandbox, one filter rejected 61,934 of 75,259 input event rows while another rejected 0; RF identifiers and counts can differ across runs or environments.
3. `RowsProduced` under the `events_modelled` Scan is the number of rows it passed onward. Compare it with `ProbeRows` under `HASH_JOIN_OPERATOR`. In the tested run, both were 13,325, matching the query result because the remaining product identifiers each matched one dimension row.
4. `ScanRows` counts rows scanned by the probe-side Scan. It was 75,259 in the tested run, so `RF... FilterRows` must **not** be read as a count of rows Doris never read from storage. This Profile demonstrates fewer rows reaching the Join, not a specific amount of disk I/O saved.

The query result is determined by the table data and SQL. The Runtime Filter identifiers, type, timing, and per-filter counts are execution details, so use the displayed Profile rather than treating the sample numbers as universal expected values.


### Stop the Doris sandbox

Run this optional cell to release CPU and memory. Docker named volumes preserve `events_modelled`, `dim_products`, and the controlled Join tables.


In [9]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");


doris
container=exited


**Expected result:** Docker reports `container=exited`.

### Restart the Doris sandbox

Run this cell before continuing to another module. It starts the existing container, reconnects to FE, and verifies the persisted product dimension.

This restart cell is idempotent: it can be run when Docker Desktop and the container are stopped, starting, or already running. On macOS it opens Docker Desktop when necessary; on Linux, start Docker Engine before running the cell.


In [ ]:
lab.start_container("doris")

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.sql("SELECT COUNT(*) AS dimension_rows FROM dim_products", title="Recovered product dimension");


**Expected result:** the container returns to `healthy`, and `dimension_rows` is 204,231.

## Lab complete

You generated a deterministic dimension from baseline identifiers, selected Join types from row-retention requirements, observed row multiplication, controlled NULL matching, distinguished Hash Join from Nested Loop Join, read planned Join distribution and observed probe-side Runtime Filter counts, and produced a fact-dimension analytical result.

Official references: [Doris Joins](https://doris.apache.org/docs/4.x/query-data/join/) · [EXPLAIN](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-query/EXPLAIN/) · [Runtime Filter](https://doris.apache.org/docs/4.x/query-acceleration/optimization-technology-principle/runtime-filter/) · [Adjusting Join Shuffle Mode](https://doris.apache.org/docs/4.x/query-acceleration/tuning/tuning-plan/adjusting-join-shuffle/)
